# Tree-of-Thoughts: Branch and Evaluate [Step 03.03]

> **MLCourse - Agentic AI - Agent Patterns**

Self-consistency samples N **complete** answers and votes at the end. It is
willing to pay for a whole reasoning chain before finding out it was doomed at
step one.

Tree-of-Thoughts (Yao et al., 2023) evaluates *partial* reasoning and prunes:

```
                      problem
                    /    |    \
              approach approach approach      <- BRANCH (generate k options)
                 A       B        C
                7/10    3/10     9/10          <- EVALUATE (score each)
                 |                |
                 +----------------+            <- PRUNE (keep the best 2)
                 |                |
              expand           expand          <- go deeper on survivors only
```

Three operations - **branch**, **evaluate**, **prune** - applied repeatedly.

### What you'll learn

- The three operations implemented plainly, with no framework.
- Why the **evaluator** is the entire ballgame, and how to ground it.
- A measured comparison against a single pass and against self-consistency.
- An honest account of when the cost is not justified.

### Why it matters

ToT is the most over-applied pattern in this module. It is genuinely powerful
where the search space is real and partial solutions can be *checked* - puzzles,
planning, code that compiles or does not. It is expensive theatre where the task
is a single-step question, which is most tasks.

### Prerequisites

- [02_self_consistency](02_self_consistency.ipynb)
- [02_langgraph/08_advanced_reasoning_patterns](../../../02_langgraph/08_advanced_reasoning_patterns) - plan-and-execute, the non-branching cousin.

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


### 1. A task where search actually helps

The right task shape for ToT has: many possible approaches, and a way to check a
*partial* answer cheaply.

We use a small arithmetic-puzzle variant of the classic "Game of 24": given four
numbers, reach the target using each number exactly once. The key property is
that **we can verify any candidate expression with Python** - which gives the
evaluator real grounding instead of an opinion.

In [2]:
import itertools, re

NUMBERS = [3, 7, 8, 9]
TARGET = 24


def verify(expression: str):
    """Evaluate a candidate expression and check it uses exactly our numbers.

    Returns (is_valid, value, reason). This is a DETERMINISTIC checker - the
    single most important component in the whole notebook.
    """
    expr = expression.strip().rstrip(".").replace("x", "*").replace("X", "*")
    expr = expr.split("=")[0].strip()
    if not re.fullmatch(r"[0-9+\-*/() .]+", expr or ""):
        return False, None, "contains characters other than digits and operators"
    used = [int(n) for n in re.findall(r"\d+", expr)]
    if sorted(used) != sorted(NUMBERS):
        return False, None, "uses %s, must use exactly %s" % (sorted(used), sorted(NUMBERS))
    try:
        value = eval(expr, {"__builtins__": {}}, {})
    except Exception as exc:
        return False, None, "does not evaluate: %s" % exc
    return abs(value - TARGET) < 1e-6, value, "evaluates to %g" % value


# Sanity-check the checker itself before trusting it.
for e in ["(3+9)*(8-7)*2", "(9-7)*8+3", "3*8*(9-7)", "8*(9-7)+3+7"]:
    print("%-16s %s" % (e, verify(e)))

(3+9)*(8-7)*2    (False, None, 'uses [2, 3, 7, 8, 9], must use exactly [3, 7, 8, 9]')
(9-7)*8+3        (False, 19, 'evaluates to 19')
3*8*(9-7)        (False, 48, 'evaluates to 48')
8*(9-7)+3+7      (False, None, 'uses [3, 7, 7, 8, 9], must use exactly [3, 7, 8, 9]')


> **Build the checker first, and test the checker.** A ToT search steered by a
> broken evaluator explores confidently in the wrong direction and burns your
> whole budget doing it. Notice we verified the verifier above - including a
> case it must reject.

### 2. Branch

Generate k candidate *approaches* - not answers. The distinction matters: we want
diverse strategies, so we ask for them in one call with an explicit instruction
to differ.

One call producing k options is cheaper than k calls, and it also gives the model
the chance to make them genuinely different rather than accidentally identical.

In [3]:
K = 4

BRANCH_PROMPT = """Find a way to make exactly {target} using the numbers {nums},
each number exactly once, with + - * / and parentheses.

Propose {k} DIFFERENT candidate expressions. Make them structurally different from
each other - do not just reorder one idea.

Output exactly {k} lines, each in this form and nothing else:
CANDIDATE: <expression>"""


def branch(k=K, extra=""):
    txt = chat([("user", BRANCH_PROMPT.format(target=TARGET, nums=NUMBERS, k=k) + extra)],
               temperature=0.9, max_tokens=260).content
    return [m.strip() for m in re.findall(r"CANDIDATE\s*:\s*(.+)", txt)][:k]


candidates = branch()
for c in candidates:
    print(" ", c)

  (9 - 7) * (8 + 3 + 5) -- wait, 5 is not in the set. Let's restart.


### 3. Evaluate

Here is the fork in the road, and it decides whether your ToT works:

| Evaluator | Grounding | Cost | Verdict |
|---|---|---|---|
| **Deterministic checker** | Total | Free | Use it whenever it exists |
| **LLM with a rubric** | Weak | One call per node | Only when nothing else is possible |
| **LLM asked "is this good?"** | None | One call per node | Never - it says yes |

We have a checker, so we use it. We then *also* run an LLM evaluator on the same
candidates, purely to show you the difference.

In [4]:
def score_deterministic(expr):
    ok, value, reason = verify(expr)
    if ok:
        return 10, reason
    if value is None:
        return 0, reason
    return max(1, 8 - min(7, abs(value - TARGET) / 3)), reason      # closeness = partial credit


print("%-22s %6s  %s" % ("candidate", "score", "checker says"))
print("-" * 72)
scored = []
for c in candidates:
    s, why = score_deterministic(c)
    scored.append((s, c, why))
    print("%-22s %6.1f  %s" % (c[:22], s, why))

candidate               score  checker says
------------------------------------------------------------------------
(9 - 7) * (8 + 3 + 5)     0.0  contains characters other than digits and operators


In [5]:
# For contrast only: what does an ungrounded LLM evaluator say about the SAME list?
JUDGE = ("Rate each candidate expression from 0 to 10 for how likely it is to equal "
         "%d using %s exactly once each. Output one line per candidate as "
         "'<expression> -> <score>'. Do not explain." % (TARGET, NUMBERS))

judged = chat([("user", JUDGE + "\n\n" + "\n".join(candidates))],
              temperature=0.0, max_tokens=200).content.strip()
print("LLM JUDGE:\n%s" % judged)
print()
print("DETERMINISTIC CHECKER:")
for s, c, why in scored:
    print("  %-22s -> %.1f   (%s)" % (c[:22], s, why))
print()
print("Compare the two. Where they disagree, the checker is right by construction.")

LLM JUDGE:
(9 - 7) * (8 + 3 + 5) -> 0

DETERMINISTIC CHECKER:
  (9 - 7) * (8 + 3 + 5)  -> 0.0   (contains characters other than digits and operators)

Compare the two. Where they disagree, the checker is right by construction.


### 4. Prune and expand

Keep the top-scoring survivors and spend the next round only on them. Everything
else is discarded and never paid for again - that is the saving.

In [6]:
BEAM = 2                     # how many survivors we carry to the next depth
MAX_DEPTH = 3

def search(verbose=True):
    """Branch / evaluate / prune, until solved or out of depth."""
    frontier, history, llm_calls = [], [], 0
    hint = ""

    for depth in range(1, MAX_DEPTH + 1):
        cands = branch(extra=hint)
        llm_calls += 1
        scored = sorted(((score_deterministic(c)[0], c) for c in cands), reverse=True)
        frontier = scored[:BEAM]
        history.append((depth, len(cands), frontier[0][0] if frontier else 0,
                        frontier[0][1] if frontier else ""))

        if verbose:
            print("depth %d: %d candidates, best %.1f -> %s"
                  % (depth, len(cands), frontier[0][0], frontier[0][1][:30]))
            for s, c in scored[BEAM:]:
                print("         pruned %-24s (%.1f)" % (c[:24], s))

        if frontier and frontier[0][0] >= 10:
            return frontier[0][1], depth, llm_calls, history

        # Feed the failures back so the next branch does not repeat them.
        hint = ("\n\nThese were already tried and are WRONG, do not repeat them:\n"
                + "\n".join("- %s (%s)" % (c, score_deterministic(c)[1]) for _, c in scored))

    return (frontier[0][1] if frontier else ""), MAX_DEPTH, llm_calls, history


solution, depth, tot_calls, hist = search()
ok, value, why = verify(solution)
print("\nSOLUTION : %s" % solution)
print("VERIFIED : %s (%s)" % (ok, why))
print("DEPTH    : %d" % depth)
print("LLM CALLS: %d" % tot_calls)

depth 1: 4 candidates, best 6.0 -> 3 * (8 + 7 - 9)
         pruned 8 / (3 - 9 / 7)          (1.6)
         pruned (9 - 7) * (8 * 3)        (1.0)


depth 2: 4 candidates, best 10.0 -> (9 + 7 - 8) * 3
         pruned 3 * 8 * (9 - 7) / 2      (0.0)
         pruned (9 - 7) * (8 + 3 + 1)    (0.0)

SOLUTION : (9 + 7 - 8) * 3
VERIFIED : True (evaluates to 24)
DEPTH    : 2
LLM CALLS: 2


> **The `hint` feedback is what makes this a search rather than repeated
> guessing.** Without it, each depth re-samples from the same distribution and
> re-proposes the same failures. Carrying the rejected candidates forward is the
> cheapest possible form of learning within a run - and note that it is exactly
> the Reflexion idea from `02_langgraph/08`, applied inside a single task.

### 5. Compare against the cheaper options

Three ways to attack the same puzzle. Measured, not asserted.

In [7]:
calls_before = USAGE["calls"]

# (a) single pass
single = chat([("user", "Make exactly %d using %s, each exactly once, with + - * / "
                        "and parentheses. Reply with ONLY the expression."
                        % (TARGET, NUMBERS))],
              temperature=0.0, max_tokens=90).content.strip()
single_ok = verify(single)[0]

# (b) self-consistency: 4 independent attempts, take any that verifies
sc_attempts = []
for _ in range(4):
    a = chat([("user", "Make exactly %d using %s, each exactly once. Reply with ONLY "
                       "the expression." % (TARGET, NUMBERS))],
             temperature=0.8, max_tokens=90).content.strip()
    sc_attempts.append(a)
sc_ok = any(verify(a)[0] for a in sc_attempts)

print("%-24s %8s %10s   %s" % ("method", "solved", "llm calls", "output"))
print("-" * 74)
print("%-24s %8s %10d   %s" % ("single pass", single_ok, 1, single[:26]))
print("%-24s %8s %10d   %s" % ("sample 4, keep any valid", sc_ok, 4,
                               next((a for a in sc_attempts if verify(a)[0]),
                                    sc_attempts[0])[:26]))
print("%-24s %8s %10d   %s" % ("tree search", ok, tot_calls, solution[:26]))
print()
print("all sc attempts:", [a[:18] for a in sc_attempts])

method                     solved  llm calls   output
--------------------------------------------------------------------------
single pass                 False          1   (9 - 7) * (8 + 3)
sample 4, keep any valid    False          4   (9 - 7) * (8 + 3)
tree search                  True          2   (9 + 7 - 8) * 3

all sc attempts: ['(9 - 7) * (8 + 3)', '(9 - 7 / 3) * 8', '9 - 8 + 7 * 3', '(9 - 7) * (8 + 3)']


### Reading this honestly

Three outcomes are possible and all are informative:

- **Single pass already solved it.** Then ToT bought nothing on this instance,
  and the correct engineering conclusion is that the task is too easy for search.
  This happens often with modern models on Game-of-24-sized problems.
- **Sampling solved it and ToT also did.** Then the deciding factor is cost, and
  "sample 4 and keep any that verifies" is simpler *and* usually cheaper. When
  you have a verifier and the search is shallow, **best-of-n with verification
  beats tree search** - it is the same idea with less machinery.
- **Only ToT solved it.** That is the case ToT exists for: the feedback from
  rejected branches changed the distribution the next branch sampled from.

The pattern to take away is not "always use ToT". It is:

> If you have a verifier, use it. Whether you wrap it in sampling or in a tree is
> a second-order decision. **Having the verifier at all is the first-order one.**

### 6. Pitfalls

- **An ungrounded evaluator.** An LLM asked whether its own idea is promising
  says yes. Search then explores confidently and uselessly.
- **No depth limit.** Branching factor `k` at depth `d` is `k^d` calls. Cap it.
- **Not feeding failures forward.** Without the reject list you are just
  resampling.
- **Using it where there are no branches.** "What is our refund policy?" has no
  search space. ToT there is pure cost.
- **Forgetting best-of-n exists.** With a verifier and a shallow search, it is
  simpler and often cheaper.

### Recap

| Idea | Takeaway |
|---|---|
| Branch / evaluate / prune | The whole algorithm, three operations |
| The evaluator is everything | Deterministic > rubric > "is this good?" (never) |
| Feed failures forward | Otherwise each round resamples the same mistakes |
| Cap depth and beam | Cost is exponential in depth |
| Compare against best-of-n | Often simpler and cheaper for the same result |

**Next:** [04_accuracy_vs_cost](04_accuracy_vs_cost.ipynb) - all of it on one
curve, with measured tokens, latency and accuracy.